In [1]:
%pip install python-dotenv google-genai
%pip install -U "langchain[google-genai]"
%pip install -U langchain-community pypdf
%pip install chromadb
%pip install -U sentence-transformers langchain-huggingface


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Test cell to check connection to LLM
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

# Load environment variables from .env file
load_dotenv()

# Get the API key from the environment variable
# The .env file has the API KEY.
api_key = os.getenv("GEMINI_API_KEY")
model = init_chat_model("google_genai:gemini-2.5-flash-lite")

response = model.invoke("Do parrots talk")
print(response)

/home/rvv/Downloads/Resumizer/gemini-env/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/home/rvv/Downloads/Resumizer/gemini-env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


content='Yes, **parrots are well-known for their ability to "talk,"** though it\'s important to understand what that means.\n\nHere\'s a breakdown:\n\n*   **Mimicry, Not True Understanding (Usually):** When parrots "talk," they are primarily **mimicking sounds** they hear. They have a remarkable ability to replicate human speech, other bird sounds, and even environmental noises like doorbells or alarms. They don\'t typically understand the meaning of the words in the same way humans do.\n\n*   **Vocal Apparatus:** Parrots have a specialized vocal organ called the **syrinx**. This organ, located at the base of their trachea, is highly adaptable and allows them to produce a wide range of sounds.\n\n*   **Intelligence and Learning:** While it\'s mimicry, it\'s not a simple reflex. Parrots are highly intelligent birds. They learn through observation and repetition. They can learn to associate certain sounds or words with specific contexts or rewards. For example, a parrot might learn to sa

In [3]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
# Path to your main folder
path = "./data/data/data/ENGINEERING"

# glob="**/*.pdf" tells it to look in all subfolders recursively
loader = DirectoryLoader(
    path, 
    glob="**/*.pdf", 
    loader_cls=PyPDFLoader,
    show_progress=True, # Shows a handy progress bar
    use_multithreading=True # Speeds up loading for large volumes
)

docs = loader.load()
len(docs)

100%|████████████████████████████████████████████████████████████████████████████████████████████| 118/118 [00:21<00:00,  5.39it/s]


227

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    length_function=len
)

chunks = text_splitter.split_documents(docs)

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import os
import shutil

# 1. FORCE DELETE the old folder to be 100% sure
if os.path.exists("./chroma_db"):
    shutil.rmtree("./chroma_db")
# "all-MiniLM-L6-v2" is fast, small, and very accurate for its size
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
abs_path = os.path.abspath("./chroma_db")
# Now use it exactly like you did before
vectorstore = Chroma.from_documents(
    documents=chunks, 
    embedding=embeddings,
    persist_directory=abs_path
)

Loading weights: 100%|█████| 103/103 [00:00<00:00, 1100.25it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
# 3. Verify it worked
count = vectorstore._collection.count()
if(len(chunks)==count):
    print("The vector db row count and number of chunks are equal")
else:
    raise Error("Corrupt db")

The vector db row count and number of chunks are equal


In [7]:
query = "Find all candidates with experience in Computer science"
relevant_chunks = vectorstore.similarity_search(query, k=5)

for i, doc in enumerate(relevant_chunks):
    print(f"\n-------------------- Result {i+1} ---------------------------")
    print(f"Content: {doc.page_content[:400]}...") # Print first 200 chars
    print(f"Metadata: {doc.metadata}")


-------------------- Result 1 ---------------------------
Content: August 16, 1989 at International Conference on Expert Systems and Neural Networks - Theory & Applications
B. S
 
: 
Computer Science
 
, 
1988
 
Union College
 
ï¼​ 
City
 
, 
State
 
Computer Science
Skills
3D, Agile, AJAX, approach, B2B, budget, C, C++, competitive, CSS, database, delivery, e-commerce, Expert Systems, funds, hiring, HTML,
PHP, image, inspection, Java, JavaScript, Marketing, Mong...
Metadata: {'title': '', 'producer': 'Qt 4.8.7', 'page_label': '2', 'source': 'data/data/data/ENGINEERING/10624813.pdf', 'creationdate': '2021-08-08T15:49:13+05:30', 'page': 1, 'creator': 'wkhtmltopdf 0.12.4', 'total_pages': 2}

-------------------- Result 2 ---------------------------
Content: client engagements
Adjunct Professor
Â Claremont Graduate University, Claremont, CA
Designed and Co-Facilitated a new transdisciplinary course "The Art & Science of Computational Thinking for Industry" for Masters and
PhD students. Â

In [11]:
import os
from langchain.chat_models import init_chat_model
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

llm = init_chat_model("google_genai:gemini-2.5-flash-lite",temperature=0)

# 2. Convert your existing Chroma vector store into a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# 3. Define the system prompt
system_prompt = (
    "You are an assistant for recruiting tasks. "
    "Use the following pieces of retrieved resumes to answer the question. "
    "If you don't know the answer, say that you don't know. "
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

# 4. Create the helper chain that "stuffs" the docs into the prompt
question_answer_chain = create_stuff_documents_chain(llm, prompt)

# 5. Create the final retrieval chain
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

response = rag_chain.invoke({"input": "Which candidates have more than 3 years of experience?"})

print(response["answer"])

Based on the provided resumes, here are the candidates with more than 3 years of experience:

*   **Software Engineering Manager:** Has 15+ years of experience.
*   **MANAGER, QUALITY ENGINEERING:** Has over 14 years of experience, with the last 8 years in management.
